# 40 · Reasoning effort and thinking across tool turns
Level L4 · Part 5 · professional · ~35 min · builds on 12, 35

## Goal
Run Scout on 10 questions at low and high reasoning effort and print a table of tokens, cost, latency
and accuracy. Show that thinking tokens are billed as output, and keep the model's thinking state
(Gemini thought signatures, Anthropic thinking blocks) intact across tool turns, including through a
save to JSON.

## The idea
Current models can **think** before they answer: they write hidden reasoning tokens, then the reply.
You don't set a token budget any more. You set a **reasoning effort**: a dial from "barely think" to
"think hard". Gemini calls it `thinking_level`, Anthropic `effort`, OpenAI `reasoning.effort`. You
never see the raw thinking, but you **pay for every thinking token at the output price**. In an agent,
the thinking behind a tool call has to come back on the next call. The vendor gives you an encrypted
copy (a **thought signature** or a **thinking block**), and you must send it back unchanged.

Analogy: a consultant who bills for thinking time too, and keeps their working notes in a sealed
envelope. You pay for the hours. You hand the envelope back at every meeting, and you don't open it.

## Picture
```mermaid
flowchart LR
    Q["question"] --> C["call 1<br/>effort = low or high"]
    C --> T["thinking tokens<br/>(hidden, billed as output)"]
    T --> F["function_call<br/>+ thought_signature"]
    F --> R["run tool"]
    R --> C2["call 2: history with the<br/>model turn sent back unchanged"]
    C2 --> A["answer"]
```

## Step by step
### 1. Start the recorder
Every model call is recorded once to `../data/llm_cassettes/40_reasoning-and-thinking.yaml` and
replayed after that, so the notebook runs with no key (lesson 07).

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("40_reasoning-and-thinking")

LLM mode: replay (40_reasoning-and-thinking.yaml)


### 2. Setup from lessons 12 and 35
The client, the pinned model and Scout's notes (lesson 35), and the price table with `cost_usd`
(lesson 12). Nothing new.

In [2]:
import ast, json, operator, os, pathlib, time
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv("../.env")
MODEL = os.environ.get("TUT_LLM_MODEL", "gemini-3.5-flash-lite")
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")
NO_AFC = types.AutomaticFunctionCallingConfig(disable=True)
NOTES = [
    {"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
    {"id": 2, "text": "Venus spins backwards compared with most planets."},
    {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
    {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
    {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
    {"id": 6, "text": "Saturn's rings are mostly made of ice."},
]
BRIEFING = {"id": 7, "text": "Mars mission briefing: " + "check suits, check air, check water. " * 40}
LOG = [{"id": 100 + d, "text": f"Mars habitat log, day {d}: routine checks, all systems normal."} for d in range(1, 31)]
ALL_NOTES = NOTES + [BRIEFING] + LOG
PRICES_PER_MTOK = {"gemini-3.5-flash-lite": {"input": 0.30, "output": 2.50}}   # USD, Gemini pricing page, 2026-10-06

def cost_usd(model: str, input_tokens: int, output_tokens: int) -> float:
    price = PRICES_PER_MTOK[model]
    return (input_tokens * price["input"] + output_tokens * price["output"]) / 1_000_000

print(len(ALL_NOTES), "notes | model:", MODEL)

37 notes | model: gemini-3.5-flash-lite


Scout's two read-only tools from lesson 35, condensed: `search_notes` with pages, and the safe
`calculator`. Their schemas are written out by hand here instead of by lesson 27's registry.

In [3]:
def search_notes(query: str, max_results: int = 5, page: int = 1) -> dict:
    q = query.strip().lower()
    if not q or page < 1:
        raise ValueError("query must not be empty and page starts at 1, e.g. query='Mars', page=1")
    hits = [n for n in ALL_NOTES if q in n["text"].lower()]
    size, start = min(max(max_results, 1), 10), (page - 1) * min(max(max_results, 1), 10)
    return {"total": len(hits), "page": page,
            "results": [{"id": n["id"], "text": n["text"][:120]} for n in hits[start:start + size]],
            "next_page": page + 1 if start + size < len(hits) else None}

OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
       ast.Div: operator.truediv, ast.Mod: operator.mod, ast.Pow: operator.pow}
def evaluate(node):
    if isinstance(node, ast.Constant) and type(node.value) in (int, float):
        return node.value
    if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.USub):
        return -evaluate(node.operand)
    if isinstance(node, ast.BinOp) and type(node.op) in OPS:
        return OPS[type(node.op)](evaluate(node.left), evaluate(node.right))
    raise ValueError("not plain arithmetic: use numbers, + - * / % ** and brackets, e.g. '24*60+39'")

def calculator(expression: str) -> float:
    return evaluate(ast.parse(expression, mode="eval").body)

print(search_notes("day on Mars")["results"], calculator("24*60+39"))

[{'id': 3, 'text': 'A day on Mars lasts about 24 hours and 39 minutes.'}] 1479


The schemas, and lesson 35's `run_tool`, which never raises.

In [4]:
SCHEMAS = [
    {"name": "search_notes", "description": "Search the user's notes for a word or phrase (case-insensitive). "
     "Returns one page of {id, text}, the total number of matches and next_page.",
     "parameters": {"type": "object", "required": ["query"], "properties": {
         "query": {"type": "string"}, "max_results": {"type": "integer"}, "page": {"type": "integer"}}}},
    {"name": "calculator", "description": "Evaluate arithmetic: numbers, + - * / % ** and brackets.",
     "parameters": {"type": "object", "required": ["expression"],
                    "properties": {"expression": {"type": "string"}}}},
]
TOOLS = {"search_notes": search_notes, "calculator": calculator}

def run_tool(call: dict, tools: dict) -> dict:
    fn = tools.get(call["name"])
    if fn is None:
        return {"content": f"unknown tool '{call['name']}'. Available: {', '.join(tools)}.", "is_error": True}
    try:
        return {"content": json.dumps(fn(**call["args"])), "is_error": False}
    except Exception as e:
        return {"content": f"{type(e).__name__}: {e}", "is_error": True}

print(run_tool({"name": "calculator", "args": {"expression": "1479/0"}}, TOOLS))

{'content': 'ZeroDivisionError: division by zero', 'is_error': True}


Lesson 35's loop, with a `verbose` switch so the benchmark can run quietly.

In [5]:
def run_agent(question: str, model, tools: dict, schemas: list[dict], max_steps: int = 8,
              verbose: bool = True) -> dict:
    messages = [{"role": "user", "text": question}]
    for step in range(1, max_steps + 1):
        reply = model(messages, schemas)
        messages.append(reply)
        if not reply.get("tool_calls"):
            return {"answer": reply["text"], "stop": "end_of_turn", "steps": step, "messages": messages}
        for call in reply["tool_calls"]:
            result = run_tool(call, tools)
            if verbose:
                print(f"step {step}: {call['name']}({call['args']}) -> {result['content'][:60]}")
            messages.append({"role": "tool", "call_id": call["id"], "name": call["name"], **result})
    return {"answer": None, "stop": "step_limit", "steps": max_steps, "messages": messages}

Lesson 35's conversion to Gemini's format. The important line: a model turn goes back as the `raw`
content Gemini returned, never rebuilt.

In [6]:
def declarations(schemas: list[dict]) -> list[types.Tool]:
    return [types.Tool(function_declarations=[types.FunctionDeclaration(
        name=s["name"], description=s["description"], parameters_json_schema=s["parameters"]) for s in schemas])]

def to_contents(messages: list[dict]) -> list[types.Content]:
    contents = []
    for m in messages:
        if m["role"] == "user":
            contents.append(types.Content(role="user", parts=[types.Part(text=m["text"])]))
        elif m["role"] == "assistant":
            contents.append(m["raw"])                          # keeps thought signatures
        else:
            response = {"error": m["content"]} if m["is_error"] else {"output": json.loads(m["content"])}
            part = types.Part(function_response=types.FunctionResponse(id=m["call_id"], name=m["name"],
                                                                       response=response))
            if contents[-1].role == "user" and contents[-1].parts[0].function_response:
                contents[-1].parts.append(part)
            else:
                contents.append(types.Content(role="user", parts=[part]))
    return contents

print("setup done")

setup done


### 3. The smallest example: one question, two effort levels
Gemini's dial is `ThinkingConfig(thinking_level=...)`. On `gemini-3.5-flash-lite` it takes `MINIMAL`
(its default), `LOW`, `MEDIUM` or `HIGH`. We ask one arithmetic question with no tools, at `LOW` and at
`HIGH`, and time each call.

One detail for timing: while recording, the course's recorder spaces calls 4 seconds apart. `timed`
waits out that gap first, so the clock measures only the model.

In [7]:
def timed(call):
    if not llm_replay.is_replaying():           # recording or live: skip llm_replay's 4 s spacing
        time.sleep(float(os.environ.get("TUT_LLM_MIN_INTERVAL", "4")))
    t0 = time.perf_counter()
    result = call()
    return result, time.perf_counter() - t0

def thinking(level: str, include_thoughts: bool = False) -> types.ThinkingConfig:
    return types.ThinkingConfig(thinking_level=level, include_thoughts=include_thoughts)

PUZZLE = ("A day on Mars lasts 24 hours and 39 minutes. How many seconds are there in 3 Mars days? "
          "Reply with the number only.")
print("expected:", 3 * (24 * 3600 + 39 * 60))

expected: 266220


Two real calls. We also ask for thought summaries (`include_thoughts=True`), to look at in step 4.

In [8]:
single = {}
for level in ["LOW", "HIGH"]:
    cfg = types.GenerateContentConfig(max_output_tokens=4000, thinking_config=thinking(level, True))
    resp, seconds = timed(lambda: client.models.generate_content(model=MODEL, contents=PUZZLE, config=cfg))
    u = resp.usage_metadata
    single[level] = {"resp": resp, "seconds": seconds, "in": u.prompt_token_count,
                     "out": u.candidates_token_count, "thoughts": u.thoughts_token_count or 0}
    print(f"{level:4}: answer {resp.text.strip():>8} | in {u.prompt_token_count} | visible out "
          f"{u.candidates_token_count} | thinking {u.thoughts_token_count}")

LOW : answer   266220 | in 32 | visible out 6 | thinking 231
HIGH: answer   266220 | in 32 | visible out 6 | thinking 653


Both levels got 266220 right. The visible answer is a handful of tokens either way. The thinking is
not: `HIGH` wrote nearly three times as many thinking tokens as `LOW`.

Timings only mean something from the recording run: a replay answers instantly. So `keep_recorded`
saves real numbers to `../data/recorded/40.json` while recording, and returns the saved ones on a
replay, or when the model cell was skipped.

In [9]:
RECORDED = pathlib.Path("../data/recorded/40.json")

def keep_recorded(key: str, value):
    data = json.loads(RECORDED.read_text()) if RECORDED.exists() else {}
    if value is not None and not llm_replay.is_replaying():          # recording run: save real numbers
        data[key] = value
        RECORDED.write_text(json.dumps(data))
    return data[key]                                                  # recorded output

fresh = {lv: {k: r[k] for k in ("in", "out", "thoughts", "seconds")} for lv, r in single.items()} \
    if "single" in globals() else None
usage = keep_recorded("single", fresh)
print({lv: f"{u['seconds']:.1f} s" for lv, u in usage.items()})

{'LOW': '2.6 s', 'HIGH': '7.3 s'}


Now the bill, the right way and the wrong way.

In [10]:
for level, r in usage.items():
    right = cost_usd(MODEL, r["in"], r["out"] + r["thoughts"])     # thinking billed as output
    wrong = cost_usd(MODEL, r["in"], r["out"])                     # forgets the thinking
    print(f"{level:4}: {r['seconds']:.1f} s | billed ${right:.6f} | without thinking "
          f"${wrong:.6f} | thinking is {1 - wrong / right:.0%} of the bill")

LOW : 2.6 s | billed $0.000602 | without thinking $0.000025 | thinking is 96% of the bill
HIGH: 7.3 s | billed $0.001657 | without thinking $0.000025 | thinking is 99% of the bill


Thinking is almost the whole bill for a short answer. A cost tracker that reads only
`candidates_token_count` (the visible reply) would under-report this call many times over.

### 4. What comes back: summaries and signatures
With `include_thoughts=True`, the reply has extra parts marked `thought=True`. They hold a **summary**
of the thinking, not the thinking itself. The answer part carries a `thought_signature`: opaque bytes,
an encrypted record of the reasoning.

In [11]:
for part in single["HIGH"]["resp"].candidates[0].content.parts:
    kind = "THOUGHT SUMMARY" if part.thought else "answer"
    sig = f"signature {len(part.thought_signature)} bytes" if part.thought_signature else "no signature"
    print(f"{kind:15} | {sig:20} | {(part.text or '').strip()[:70]!r}")
print("resp.text:", repr(single["HIGH"]["resp"].text))

THOUGHT SUMMARY | no signature         | '**My Thought Process for Calculating Seconds in 3 Mars Days**\n\nAlright'
answer          | signature 1387 bytes | '266220'
resp.text: '266220'


`resp.text` skips the summary parts, so Scout's answer stays clean. The summary is written for people;
what you are billed for is `thoughts_token_count`, the thinking the model actually did. Both Gemini
and Anthropic return summaries only, never the raw chain of thought.

### 5. The same dial at three vendors
The names differ; the idea is the same. Values are from RESEARCH.md and the vendors' docs (checked
2026-10-09).

| | Gemini 3.x | Anthropic (Claude 4.7+) | OpenAI (GPT-6) |
|---|---|---|---|
| dial | `thinking_config.thinking_level` | `output_config.effort` | `reasoning.effort` |
| values | minimal, low, medium, high (per model) | low, medium, high, xhigh, max | none, low, medium, high, xhigh, max |
| old knob | `thinking_budget` | `budget_tokens` (400 on Claude 4.7+) | |
| billed as | output (`thoughts_token_count`) | output (`output_tokens`) | output (reasoning tokens) |
| state to send back | `thought_signature` on parts | `thinking` blocks with `signature` | `reasoning` output items |

Defaults differ too: `gemini-3.5-flash-lite` defaults to minimal, Claude Sonnet 5.5 to high. Always
set the level yourself so a model swap doesn't silently change your bill.

### 6. Scout with an effort level
Lesson 35's `GeminiModel`, plus a `level` and one row per call: input, visible output, thinking and
seconds. `max_output_tokens` is generous because thinking counts toward it.

In [12]:
SYSTEM = ("You are Scout. Answer questions about the user's notes. Search the notes first. "
          "Use the calculator for arithmetic. If a tool returns an error, fix the call and try again. "
          "Answer in one sentence.")

class GeminiModel:
    def __init__(self, level: str, include_thoughts: bool = False, max_output_tokens: int = 6000):
        self.level, self.include_thoughts, self.max_output_tokens = level, include_thoughts, max_output_tokens
        self.calls = []                                   # one usage row per model call

    def __call__(self, messages: list[dict], schemas: list[dict]) -> dict:
        cfg = types.GenerateContentConfig(system_instruction=SYSTEM, max_output_tokens=self.max_output_tokens,
                                          tools=declarations(schemas), automatic_function_calling=NO_AFC,
                                          thinking_config=thinking(self.level, self.include_thoughts))
        resp, seconds = timed(lambda: client.models.generate_content(
            model=MODEL, contents=to_contents(messages), config=cfg))
        u = resp.usage_metadata
        self.calls.append({"in": u.prompt_token_count or 0, "out": u.candidates_token_count or 0,
                           "thoughts": u.thoughts_token_count or 0, "seconds": seconds})
        raw = resp.candidates[0].content
        calls = [{"id": fc.id, "name": fc.name, "args": dict(fc.args)} for fc in resp.function_calls or []]
        if calls:
            return {"role": "assistant", "tool_calls": calls, "raw": raw}
        return {"role": "assistant", "text": resp.text or "", "raw": raw}

### 7. Thinking across tool turns
One question at `HIGH`, with summaries on, so we can see everything Gemini puts in each model turn.

In [13]:
traced = GeminiModel("HIGH", include_thoughts=True)
trace_run = run_agent("How many hours are 10 Mars days?", traced, TOOLS, SCHEMAS)
print("answer:", trace_run["answer"])
for i, m in enumerate(trace_run["messages"]):
    if m["role"] == "assistant":
        kinds = ["thought" if p.thought else "call" if p.function_call else "text" for p in m["raw"].parts]
        signed = ["signed" if p.thought_signature else "-" for p in m["raw"].parts]
        print(f"message {i}: parts {kinds} | {signed}")

step 1: search_notes({'query': 'Mars'}) -> {"total": 34, "page": 1, "results": [{"id": 1, "text": "Mars
step 2: search_notes({'query': 'sol'}) -> {"total": 1, "page": 1, "results": [{"id": 4, "text": "Jupit
step 3: calculator({'expression': '24.65 * 10'}) -> 246.5
answer: Based on the notes stating that a day on Mars lasts about 24 hours and 39 minutes, 10 Mars days are equal to 246.5 hours.
message 1: parts ['thought', 'call'] | ['-', 'signed']
message 3: parts ['call'] | ['signed']
message 5: parts ['call'] | ['signed']
message 7: parts ['text'] | ['signed']


Only the first turn came with a thought summary here; the later turns acted straight away. But every
part that acts (each function call, and the final text) carries a signature. Call 2 could continue call 1's reasoning only because `to_contents`
sent call 1's `raw` turn back exactly as received. Gemini checks the signatures on function calls in
the current turn; rebuild a call without one and you get lesson 22's 400 "missing a thought_signature".

### 8. Keeping thinking state through a save
Lesson 39 saved runs to disk so they could resume. A signature is `bytes`, which plain `json.dumps`
cannot write, and a hand-made `{"name", "args"}` copy of the call drops it. Pydantic's JSON mode
encodes the bytes as base64 text, and `model_validate` turns them back into the same bytes.

In [14]:
def save_turn(content: types.Content) -> str:
    return json.dumps(content.model_dump(mode="json", exclude_none=True))

def load_turn(text: str) -> types.Content:
    return types.Content.model_validate(json.loads(text))

first_turn = trace_run["messages"][1]["raw"] if "trace_run" in globals() else types.Content(
    role="model", parts=[types.Part(text="(skipped)", thought_signature=b"\x01demo")])   # stand-in if skipped
restored = load_turn(save_turn(first_turn))
print("saved size:", len(save_turn(first_turn)), "chars")
print("signatures identical after the round trip:",
      [p.thought_signature for p in restored.parts] == [p.thought_signature for p in first_turn.parts])

saved size: 2384 chars
signatures identical after the round trip: True


The restored turn is byte-for-byte the same, so a resumed run can send it back and the model picks up
its own reasoning. Never re-create a model turn from your logs; store the vendor's turn.

### 9. The same rule at Anthropic, by hand
No Anthropic key was used to build this course, so this is the documented shape, not a recorded call.
A Claude reply with a tool call holds a `thinking` block (its `thinking` text is empty by default on
Claude 5.x, `display: "omitted"`) and a `tool_use` block. The next request must repeat the assistant's
**whole** content list, thinking first, then add the `tool_result`.

In [15]:
claude_reply = [   # documented shape of response.content (platform.claude.com, "Thinking"); signature shortened
    {"type": "thinking", "thinking": "", "signature": "EosnCkYICxIMMb3LzNrMu..."},
    {"type": "tool_use", "id": "toolu_01", "name": "search_notes", "input": {"query": "day on Mars"}},
]
next_request = {
    "model": "claude-sonnet-5-5", "max_tokens": 16000, "output_config": {"effort": "low"},
    "messages": [
        {"role": "user", "content": "How many minutes long is a day on Mars?"},
        {"role": "assistant", "content": claude_reply},                    # unchanged, thinking included
        {"role": "user", "content": [{"type": "tool_result", "tool_use_id": "toolu_01",
                                      "content": json.dumps(search_notes("day on Mars"))}]},
    ],
}
print(json.dumps(next_request["messages"][1], indent=1)[:400])

{
 "role": "assistant",
 "content": [
  {
   "type": "thinking",
   "thinking": "",
   "signature": "EosnCkYICxIMMb3LzNrMu..."
  },
  {
   "type": "tool_use",
   "id": "toolu_01",
   "name": "search_notes",
   "input": {
    "query": "day on Mars"
   }
  }
 ]
}


Anthropic's rules: within a tool-use turn, pass the thinking blocks back complete and unmodified (an
edited block gets a 400); across turns, pass everything back and the API decides which old blocks to
keep. On Opus 4.5 and models from 4.6 on, kept blocks are billed as **input** on later calls.

### 10. The benchmark: 10 questions, low vs high
Ten questions over Scout's notes, each with the strings a correct answer must contain. Most need a
search and a calculation.

In [16]:
BENCH = [
    ("How many minutes long is a day on Mars?", ["1479"]),
    ("What are the names of the moons of Mars?", ["Phobos", "Deimos"]),
    ("How many hours are 10 Mars days?", ["246.5"]),
    ("How many seconds long is a Mars day?", ["88740"]),
    ("Which planet spins backwards?", ["Venus"]),
    ("What are Saturn's rings mostly made of?", ["ice"]),
    ("How many habitat log entries do I have?", ["30"]),
    ("How many of my notes mention Mars?", ["34"]),
    ("How many whole Mars days fit in 1000 Earth hours?", ["40"]),
    ("In what year did Perseverance land, and how many years before 2030 is that?", ["2021", "9"]),
]
def is_correct(answer: str | None, expected: list[str]) -> bool:
    text = (answer or "").replace(",", "").lower()
    return all(e.lower() in text for e in expected)

print(len(BENCH), "questions |", is_correct("A Mars day is 1,479 minutes.", ["1479"]))

10 questions | True


A helper runs every question at one level and keeps each question's usage rows.

In [17]:
def bench(level: str) -> list[dict]:
    model, rows = GeminiModel(level), []
    for question, expected in BENCH:
        first = len(model.calls)
        run = run_agent(question, model, TOOLS, SCHEMAS, verbose=False)
        rows.append({"question": question, "correct": is_correct(run["answer"], expected),
                     "answer": run["answer"], "calls": model.calls[first:]})
        print(f"{level:4} {'ok ' if rows[-1]['correct'] else 'BAD'} {len(rows[-1]['calls'])} calls "
              f"| {sum(c['thoughts'] for c in rows[-1]['calls']):5} thinking | {question[:45]}")
    return rows

Low effort first.

In [18]:
results = {"LOW": bench("LOW")}

LOW  ok  3 calls |     0 thinking | How many minutes long is a day on Mars?
LOW  ok  3 calls |     0 thinking | What are the names of the moons of Mars?
LOW  ok  4 calls |     0 thinking | How many hours are 10 Mars days?


LOW  ok  4 calls |     0 thinking | How many seconds long is a Mars day?
LOW  ok  3 calls |     0 thinking | Which planet spins backwards?
LOW  ok  2 calls |     0 thinking | What are Saturn's rings mostly made of?


LOW  ok  2 calls |     0 thinking | How many habitat log entries do I have?


LOW  ok  3 calls |     0 thinking | How many of my notes mention Mars?
LOW  ok  5 calls |     0 thinking | How many whole Mars days fit in 1000 Earth ho
LOW  ok  3 calls |     0 thinking | In what year did Perseverance land, and how m


Then high effort, same questions.

In [19]:
results["HIGH"] = bench("HIGH")

HIGH ok  3 calls |   474 thinking | How many minutes long is a day on Mars?
HIGH ok  2 calls |   265 thinking | What are the names of the moons of Mars?
HIGH ok  4 calls |   561 thinking | How many hours are 10 Mars days?


HIGH ok  5 calls |   709 thinking | How many seconds long is a Mars day?
HIGH ok  2 calls |   200 thinking | Which planet spins backwards?
HIGH ok  2 calls |   191 thinking | What are Saturn's rings mostly made of?
HIGH ok  2 calls |   289 thinking | How many habitat log entries do I have?
HIGH ok  2 calls |   206 thinking | How many of my notes mention Mars?


HIGH ok  5 calls |   818 thinking | How many whole Mars days fit in 1000 Earth ho


HIGH ok  3 calls |   235 thinking | In what year did Perseverance land, and how m


As in step 3, `keep_recorded` saves the real timings while recording and supplies them on a replay. If
you skipped the model cells, the whole recorded result is used.

In [20]:
saved = keep_recorded("bench", results if "results" in globals() and len(results) == 2 else None)
if "results" not in globals() or len(results) < 2:
    results = saved
for level in results:                                             # replay: use the recorded seconds
    for row, old in zip(results[level], saved[level]):
        for call, old_call in zip(row["calls"], old["calls"]):
            call["seconds"] = old_call["seconds"]
print({level: sum(r["correct"] for r in rows) for level, rows in results.items()})

{'LOW': 10, 'HIGH': 10}


### 11. The table
One line per level: accuracy, model calls, tokens, cost (thinking billed as output) and seconds per
question.

In [21]:
def summary(rows: list[dict]) -> dict:
    calls = [c for r in rows for c in r["calls"]]
    tin, tout, tthink = (sum(c[k] for c in calls) for k in ("in", "out", "thoughts"))
    per_q = [sum(c["seconds"] for c in r["calls"]) for r in rows]
    return {"correct": sum(r["correct"] for r in rows), "calls": len(calls), "in": tin, "out": tout,
            "thinking": tthink, "cost": cost_usd(MODEL, tin, tout + tthink),
            "think_cost": cost_usd(MODEL, 0, tthink), "mean_s": sum(per_q) / len(per_q), "max_s": max(per_q)}

print(f"{'level':5} {'right':>5} {'calls':>5} {'input':>6} {'output':>6} {'thinking':>8} {'cost $':>9} "
      f"{'thinking $':>10} {'s/q':>5} {'max s':>5}")
table = {level: summary(rows) for level, rows in results.items()}
for level, s in table.items():
    print(f"{level:5} {s['correct']:>3}/10 {s['calls']:>5} {s['in']:>6} {s['out']:>6} {s['thinking']:>8} "
          f"{s['cost']:>9.5f} {s['think_cost']:>10.5f} {s['mean_s']:>5.1f} {s['max_s']:>5.1f}")

level right calls  input output thinking    cost $ thinking $   s/q max s
LOW    10/10    32   7013    611        0   0.00363    0.00000   1.8   2.8
HIGH   10/10    30  11788    586     3948   0.01487    0.00987   3.0   6.1


Three things stand out. At `LOW` the model wrote **no** thinking tokens at all inside the tool loop:
Gemini's thinking is dynamic, and the level is a ceiling on effort, not a quota. `HIGH` wrote about
4,000 thinking tokens, two thirds of its bill, and took longer per question. And `HIGH` used more
input tokens with fewer calls: the only extra thing it sends is its own model turns, carrying the
thinking state. Now the "right" column. Did the levels disagree on any question?

In [22]:
for (q, _), low, high in zip(BENCH, results["LOW"], results["HIGH"]):
    if low["correct"] != high["correct"]:
        print(f"disagree: {q}\n  LOW : {low['answer']}\n  HIGH: {high['answer']}")
ratio = table["HIGH"]["cost"] / table["LOW"]["cost"]
print(f"HIGH costs x{ratio:.1f} of LOW for {table['HIGH']['correct'] - table['LOW']['correct']:+d} correct answers")

HIGH costs x4.1 of LOW for +0 correct answers


Both levels got all 10 right, so here `HIGH` bought nothing for four times the price. That is the
decision in one line: pick the lowest effort that passes your evals (lesson 48), and raise it only
for the questions that need it. Ten questions is a small sample; lesson 45 benchmarks
Scout properly.

## At work
- **Effort is tuned per route, not per app.** Teams run evals at each level and send easy requests to
  low effort or a small model, hard ones to high; it is a cost lever next to caching and batch.
  → lessons 45 and 54
- **Cost dashboards that read only visible output under-report reasoning models.** The OpenTelemetry
  GenAI conventions have a separate `gen_ai.usage.reasoning.output_tokens` field for this, and budgets
  must count it. → lesson 54
- **Changing the thinking setting breaks the prompt cache.** Anthropic documents that switching effort
  or thinking mode between requests invalidates cached message blocks, so keep it fixed within a
  conversation. → lesson 41
- **Model upgrades change the thinking API.** Claude 4.7+ rejects `budget_tokens` with a 400, and
  defaults differ per model (minimal on Gemini 3.5 Flash-Lite, high on Sonnet 5.5), so a model swap
  can change both behaviour and bill. → lesson 53

## What just happened
- One puzzle at `LOW` and `HIGH`: same right answer, 6 visible tokens, but 231 vs 653 thinking
  tokens, which were 96% and 99% of the bill.
- `include_thoughts=True` returned summary parts (`thought=True`); `resp.text` skipped them; the acting
  part carried a `thought_signature`.
- In a multi-step run every model turn went back unchanged through `raw`, so each call continued the
  previous call's reasoning.
- Pydantic's JSON mode saved and restored a turn with identical signature bytes.
- Anthropic's equivalent is the `thinking` block with a `signature`, repeated unchanged before the
  `tool_result`.
- The 10-question benchmark: both levels 10/10; `LOW` thought 0 tokens in the loop, `HIGH` about 4,000,
  costing about 4 times as much and taking 3.0 s instead of 1.8 s per question.

## Common mistakes
**1. A tight output cap with high effort.** Thinking counts toward `max_output_tokens`. With a 60-token
cap, `HIGH` spends the whole cap thinking and there is no text left (`finish_reason` `MAX_TOKENS`).
You are still billed for the thinking.

In [23]:
cfg = types.GenerateContentConfig(max_output_tokens=60, thinking_config=thinking("HIGH"))
resp = client.models.generate_content(model=MODEL, contents=PUZZLE, config=cfg)
print(resp.candidates[0].finish_reason, "| thinking:", resp.usage_metadata.thoughts_token_count)
print("answer:", resp.text.strip())

FinishReason.MAX_TOKENS | thinking: 57


AttributeError: 'NoneType' object has no attribute 'strip'

Fix: keep the cap generous (Scout uses 6000) and control cost with the effort level instead, which is
what Google's thinking guide recommends. Check `finish_reason` before reading the text.

**2. Leaving thinking out of the token sum.** Input plus visible output does not add up to the total
the response reports. A self-check catches it:

In [24]:
r = usage["HIGH"]                                   # step 3's HIGH call
total = r["in"] + r["out"] + r["thoughts"]          # what usage_metadata.total_token_count reports
billed = r["in"] + r["out"]                         # wrong: forgets thoughts_token_count
assert billed == total, f"counted {billed} tokens but the response used {total}: thinking is missing"

AssertionError: counted 38 tokens but the response used 691: thinking is missing

Fix: output = `candidates_token_count + thoughts_token_count`, as lesson 12's `usage_tokens` does,
and assert the parts add up to `total_token_count` in your logging.

## Try it
No new model calls: use `results` from step 10.
1. For each level, compute the cost per **correct** answer and the share of the bill that is thinking.
2. Scout will answer 5,000 questions a day like these. Project the monthly (30-day) cost at `LOW` and
   at `HIGH`.
3. Write `check_resent(previous_reply, next_messages)` for the Anthropic format: it raises `ValueError`
   unless the last assistant message's content equals `previous_reply` exactly. Test it on
   `next_request` from step 9, and on a copy where the thinking block was dropped.

In [25]:
# your code here

Finally, stop the recorder (in record mode this saves the recording).

In [26]:
llm_replay.stop()

## Key terms
- **reasoning effort** — the setting that controls how much a model thinks before answering; Gemini `thinking_level`, Anthropic `effort`, OpenAI `reasoning.effort`. It replaced fixed thinking budgets.
- **thought summary** — a readable summary of the model's hidden thinking, returned only when asked (`include_thoughts=True` on Gemini, `display: "summarized"` on Anthropic); never the raw reasoning.
- **thinking block** — Anthropic's content block (`type: "thinking"`, `thinking`, `signature`) that holds a reply's reasoning; send it back unchanged within a tool-use turn.
- **thinking block preservation** — the rule that a model's thinking state (Gemini signatures, Anthropic thinking blocks, OpenAI reasoning items) goes back to the API unmodified on the next call of a tool loop.